# Deception Classifier — Colab GPU Training
Run each cell in order. Make sure the runtime is set to a GPU of your choice 
Runtime → Change runtime type → i.e., A100 GPU

## 1. Clone repo & install dependencies

In [1]:
!git clone https://github.com/Lucca878/deceptionClassifierTraining.git
%cd deceptionClassifierTraining

Cloning into 'deceptionClassifierTraining'...
remote: Enumerating objects: 181, done.
remote: Counting objects: 100% (181/181), done.
remote: Compressing objects: 100% (157/157), done.
remote: Total 181 (delta 17), reused 178 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (181/181), 4.31 MiB | 13.79 MiB/s, done.
Resolving deltas: 100% (17/17), done.
/content/deceptionClassifierTraining


In [2]:
!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 150.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 93.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 7.4 MB/s eta 0:00:00


## 2. Verify GPU

In [3]:
import torch
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

GPU available: True
Device: NVIDIA A100-SXM4-40GB


## 2.5 (Optional) Set hyperparameters to test
Set these BEFORE running CV. Leave as `None` to use defaults. Same parameters will be used for both CV and full training.

In [ ]:
# Set hyperparameters to test (None = use defaults)
EPOCHS = None          # int: number of epochs (default: 2)
LEARNING_RATE = None   # float: learning rate (default: 5e-5)
BATCH_SIZE = None      # int: batch size (default: 32)
WEIGHT_DECAY = None    # float: weight decay (default: 0.01)

# Example: to test with 3 epochs and higher LR:
# EPOCHS = 3
# LEARNING_RATE = 1e-4

print("Hyperparameters to test:")
print(f"  EPOCHS: {EPOCHS if EPOCHS is not None else '(default: 2)'}")
print(f"  LEARNING_RATE: {LEARNING_RATE if LEARNING_RATE is not None else '(default: 5e-5)'}")
print(f"  BATCH_SIZE: {BATCH_SIZE if BATCH_SIZE is not None else '(default: 32)'}")
print(f"  WEIGHT_DECAY: {WEIGHT_DECAY if WEIGHT_DECAY is not None else '(default: 0.01)'}")

## 2.6 Select model

In [ ]:
MODEL_KEY = "distilbert"  # Options: distilbert, bert, sbert, modernbert
print(f"Selected model: {MODEL_KEY}")

## 3. Run cross-validation first (tuning stage)

In [ ]:
import subprocess

# Build command with optional hyperparameters
cmd = [
    "python", "src/pipeline/run_pipeline.py",
    "--mode", "cv",
    "--model", MODEL_KEY,
    "--output_root", "./outputs",
    "--seed", "42"
]

if EPOCHS is not None:
    cmd.extend(["--epochs", str(EPOCHS)])
if LEARNING_RATE is not None:
    cmd.extend(["--lr", str(LEARNING_RATE)])
if BATCH_SIZE is not None:
    cmd.extend(["--batch_size", str(BATCH_SIZE)])
if WEIGHT_DECAY is not None:
    cmd.extend(["--weight_decay", str(WEIGHT_DECAY)])

print("Running CV with command:")
print(" ".join(cmd))
print()

subprocess.run(cmd, check=True)

Training model preset: distilbert
Backbone: distilbert-base-uncased
Loaded 4542 rows from data/hippocorpus/hippocorpus_training_truncated.csv
config.json: 100% 483/483 [00:00<00:00, 2.11MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 255kB/s]
vocab.txt: 232kB [00:00, 849kB/s]
tokenizer.json: 466kB [00:00, 1.03MB/s]

--- split_1 ---
Map: 100% 3633/3633 [00:00<00:00, 4541.70 examples/s]
Map: 100% 909/909 [00:00<00:00, 6451.75 examples/s]
model.safetensors: 100% 268M/268M [00:02<00:00, 110MB/s] 
Loading weights: 100% 100/100 [00:00<00:00, 1952.90it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
c

## 4. Inspect CV results (decide hyperparameters/model)

In [5]:
import os, glob

# Find the latest CV run for the selected model
run_dirs = sorted(glob.glob(f"outputs/{MODEL_KEY}_*"))
latest = run_dirs[-1] if run_dirs else None
print("Latest run:", latest)

if latest:
    for f in os.listdir(latest):
        print(" ", f)

Latest run: outputs/distilbert_20260424_111500
  cv
  cv_results.csv
  config.json


In [6]:
import pandas as pd
import os

cv_path = os.path.join(latest, "cv_results.csv") if latest else None
if cv_path and os.path.exists(cv_path):
    # File is semicolon-separated
    df = pd.read_csv(cv_path, sep=";")
    print(df.head())
    print("\nRows, cols:", df.shape)

    # Main CV metric from stored predictions
    mean_acc = df["Correct"].mean()
    print("Mean CV accuracy:", round(mean_acc, 4))

    # Optional per-split and per-class breakdown
    print("\nAccuracy by split:")
    print(df.groupby("Split")["Correct"].mean().round(4))

    print("\nRecall by label (0=truthful, 1=deceptive):")
    print(df.groupby("Label")["Correct"].mean().round(4))

     Split  Prediction  Label  Correct
0  split_1           0      1    False
1  split_1           1      0    False
2  split_1           1      0    False
3  split_1           1      0    False
4  split_1           1      1     True

Rows, cols: (4542, 4)
Mean CV accuracy: 0.7715

Accuracy by split:
Split
split_1    0.7602
split_2    0.7602
split_3    0.7885
split_4    0.7742
split_5    0.7742
Name: Correct, dtype: float64

Recall by label (0=truthful, 1=deceptive):
Label
0    0.7330
1    0.8069
Name: Correct, dtype: float64


## 5. Train final model on full training set and zip it

In [ ]:
import os, glob, shutil, subprocess

# Fallback hyperparameters if section 2.5 wasn't run
try:
    _ = EPOCHS
except NameError:
    EPOCHS = None
try:
    _ = LEARNING_RATE
except NameError:
    LEARNING_RATE = None
try:
    _ = BATCH_SIZE
except NameError:
    BATCH_SIZE = None
try:
    _ = WEIGHT_DECAY
except NameError:
    WEIGHT_DECAY = None

# Build command with same hyperparameters used in CV
cmd = [
    "python", "src/pipeline/run_pipeline.py",
    "--mode", "full",
    "--model", MODEL_KEY,
    "--output_root", "./outputs",
    "--seed", "42"
]

if EPOCHS is not None:
    cmd.extend(["--epochs", str(EPOCHS)])
if LEARNING_RATE is not None:
    cmd.extend(["--lr", str(LEARNING_RATE)])
if BATCH_SIZE is not None:
    cmd.extend(["--batch_size", str(BATCH_SIZE)])
if WEIGHT_DECAY is not None:
    cmd.extend(["--weight_decay", str(WEIGHT_DECAY)])

print("Running full training with same hyperparameters as CV:")
print(" ".join(cmd))
print()

# Run full-data training
subprocess.run(cmd, check=True)

# Find newest full-training model folder
model_dirs = sorted(glob.glob(f"./outputs/{MODEL_KEY}_*/model", recursive=True))
if not model_dirs:
    raise FileNotFoundError(f"No trained model folder found under ./outputs/{MODEL_KEY}_*/model")

model_dir = model_dirs[-1]
zip_base = f"{MODEL_KEY}_retrained"
zip_file = zip_base + ".zip"

print("Using model dir:", model_dir)
shutil.make_archive(zip_base, "zip", model_dir)
print("Created zip:", zip_file)
print("Zip exists:", os.path.exists(zip_file), "| Size (MB):", round(os.path.getsize(zip_file) / 1024 / 1024, 2))

# Save the full path for section 6
ZIP_PATH = os.path.abspath(zip_file)
print(f"Zip file saved at: {ZIP_PATH}")

Running full training with same hyperparameters as CV:
python src/pipeline/run_pipeline.py --mode full --model distilbert --output_root ./outputs --seed 42

Using model dir: ./outputs/distilbert_20260424_113149/model
Created zip: distilbert_retrained.zip
Zip exists: True | Size (MB): 235.75


In [ ]:
from google.colab import drive
import shutil, os

# Use the path saved from section 5
src = ZIP_PATH if 'ZIP_PATH' in globals() else f"{MODEL_KEY}_retrained.zip"
assert os.path.exists(src), f"Missing file: {src}. Run section 5 first."

drive.mount("/content/drive")
dst = f"/content/drive/MyDrive/{MODEL_KEY}_trained.zip"
shutil.copy(src, dst)
print(f"Copied {src} to {dst}")
print(f"File size: {round(os.path.getsize(dst) / 1024 / 1024, 2)} MB")

Mounted at /content/drive
Copied distilbert_retrained.zip to /content/drive/MyDrive/distilbert_retrained.zip
File size: 235.75 MB
